Versuch mit dem Google Speech Commands Dataset ein CNN zu Traineren welches nachher zuverlässig Commands erkennt.
Hierfür wird  MFCC zu hilfe gezogen

Imports 


In [ ]:
import pandas as pd
import librosa
import numpy as np
import json
import sounddevice as sd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from pathlib import Path

In [ ]:
base = Path('archive') # wo sich das Dataset befindet 

Da sich es bei dem Dataset um .Wav Dateien handelt müssen diese nicht umgewandelt werden  

Dataset importieren und Trimmen, es werden nicht alle Speech Commands verwendet. (Ich nutze hier nur; up, down, left, right, on, off)

In [ ]:
# Command used 
command = {"up", "down", "left", "right", "on", "off"}

Die Daten aus dem Dataset mit MFCC umwandeln

In [ ]:
relevante_ordner = [p for p in base.iterdir() if p.is_dir() and p.name in command]


daten_liste = []


for ordner_pfad in relevante_ordner:
    label = ordner_pfad.name  
    
    for wav_datei in ordner_pfad.glob('*.wav'):
        
        try:
            y, sr = librosa.load(wav_datei, sr=16000)
            mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
            
            daten_liste.append({
                'Dateiname': wav_datei.name,
                'Dateipfad': str(wav_datei).removeprefix('archive\\'),
                'Label': label,
                'MFCC_Matrix': mfccs 
            })
            
        except Exception as e:
            print(f"Fehler beim Verarbeiten von {wav_datei.name}: {e}")

df = pd.DataFrame(daten_liste)
df.head()


Train/ Test split 


In [ ]:
testing_list = Path('archive/testing_list.txt')
validation_list = Path('archive\list.txt')

In [ ]:
bekannte_pfade = set(df['Dateipfad'])

# 1. Normale Python-Liste zum Sammeln erstellen
gefundene_eintraege_test = []
gefundene_eintraege_valid = []

# testing_list = Path('archive/testing_list.txt')

with open(testing_list, 'r', encoding='utf-8') as file:
    for line in file:
        eintrag = line.strip().replace('/', '\\')
        
        if eintrag in bekannte_pfade:
            gefundene_eintraege_test.append(eintrag)

df_test = pd.DataFrame(gefundene_eintraege_test, columns=['Dateipfad'])

# validation_list = Path('archive\validation_list.txt')

with open(validation_list, 'r', encoding='utf-8') as file:
    for line in file:
        eintrag = line.strip().replace('/', '\\')
        
        if eintrag in bekannte_pfade:
            gefundene_eintraege_valid.append(eintrag)

df_valid = pd.DataFrame(gefundene_eintraege_valid, columns=['Dateipfad'])
# Um zu prüfen, ob es geklappt hat:
print(df_test.head())
print(df_valid.head())

Die Daten wurden in train und test gesplittet, wie von google vorgegeben 


Alle MFCCs auf die gleiche Länge bringen (Padding), max_len wird sowiso für die eingabe wieder verwednet 

In [ ]:
mfcc_list = df['MFCC_Matrix'].tolist()

# Maximale Länge der Zeitachse (Achse 1, also die Spalten)
max_len = max(m.shape[1] for m in mfcc_list)
print(f"Maximale Zeitschritte: {max_len}")

# Alle Matrizen auf gleiche Länge padden (entlang Achse 1 = Zeitschritte)
X = np.array([
    np.pad(m, ((0, 0), (0, max_len - m.shape[1])), mode='constant')
    for m in mfcc_list
])

# Transponieren zu (Samples, Zeitschritte, MFCCs) – so wie CNNs es erwarten
X = X.transpose(0, 2, 1)

print(f"X Shape: {X.shape}")  # Sollte (14178, max_len, 13) sein


In [ ]:
y = df['Label'].values              # ['left', 'right', 'up', 'down', 'on', 'off']

# Labels in Zahlen umwandeln (0-5)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# One-Hot Encoding für die 6 Klassen
y_onehot = tf.keras.utils.to_categorical(y_encoded, num_classes=6)

# CNN braucht einen zusätzlichen "Kanal" (wie bei Graustufenbildern = 1 Kanal)
X = X[..., np.newaxis]  # Shape: (Samples, Zeitschritte, MFCCs, 1)

# Train/Test Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Input Shape: {X_train.shape[1:]}")
print(f"Klassen: {label_encoder.classes_}")
print(f"Training: {X_train.shape[0]} Samples, Validation: {X_val.shape[0]} Samples")

# ============================
# 2. CNN Modell definieren
# ============================

model = models.Sequential([
    # Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same', 
                  input_shape=X_train.shape[1:]),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Klassifikations-Kopf
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(6, activation='softmax')  # 6 Klassen
])

model.summary()

# ============================
# 3. Modell kompilieren & trainieren
# ============================

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Early Stopping: stoppt automatisch, wenn sich das Modell nicht mehr verbessert
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

# ============================
# 4. Ergebnisse anzeigen
# ============================

val_loss, val_acc = model.evaluate(X_val, y_val)
print(f"\nValidation Accuracy: {val_acc:.2%}")


In [ ]:
# Modell speichern
model.save('mein_cnn_modell.keras')

# Label-Mapping speichern (damit du weißt, welche Zahl welche Klasse ist)
import json
label_mapping = dict(zip(range(len(label_encoder.classes_)), label_encoder.classes_.tolist()))
with open('label_mapping.json', 'w') as f:
    json.dump(label_mapping, f)

print("Modell und Labels gespeichert!")
print(f"Label Mapping: {label_mapping}")
print(f"max_len (für Padding): {max_len}")  # Diesen Wert merken!


In [ ]:

# ============================
# Konfiguration
# ============================
DAUER = 1              # Aufnahme-Dauer in Sekunden
SAMPLERATE = 16000     # Gleiche Samplerate wie beim Training
N_MFCC = 13            # Gleiche Anzahl MFCCs wie beim Training
MAX_LEN = 32      # <-- Hier den max_len Wert aus dem Training einsetzen!

# ============================
# Modell und Labels laden
# ============================
model = tf.keras.models.load_model('licht_klassifikator_int8_basic.tflite')

with open('label_mapping.json', 'r') as f:
    label_mapping = json.load(f)

print("Modell geladen! Klassen:", label_mapping)

# ============================
# Hilfsfunktion: Audio → MFCC → Vorhersage
# ============================
def klassifiziere(audio, sr=SAMPLERATE):
    # MFCCs berechnen (gleich wie beim Training!)
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=N_MFCC)
    
    # Padding auf gleiche Länge wie beim Training
    if mfccs.shape[1] < MAX_LEN:
        mfccs = np.pad(mfccs, ((0, 0), (0, MAX_LEN - mfccs.shape[1])), mode='constant')
    else:
        mfccs = mfccs[:, :MAX_LEN]
    
    # Transponieren zu (Zeitschritte, MFCCs) und Kanal hinzufügen
    mfccs = mfccs.T  # Shape: (MAX_LEN, 13)
    mfccs = mfccs[np.newaxis, ..., np.newaxis]  # Shape: (1, MAX_LEN, 13, 1)
    
    # Vorhersage
    prediction = model.predict(mfccs, verbose=0)
    klasse_idx = np.argmax(prediction)
    klasse = label_mapping[str(klasse_idx)]
    konfidenz = prediction[0][klasse_idx]
    
    return klasse, konfidenz

# ============================
# Live-Schleife
# ============================
print("\n🎤 Live-Erkennung gestartet! (Strg+C zum Stoppen)\n")

try:
    while True:
        input(">> Enter drücken zum Aufnehmen...")
        
        # Audio vom Mikrofon aufnehmen
        audio = sd.rec(int(DAUER * SAMPLERATE), samplerate=SAMPLERATE, 
                       channels=1, dtype='float32')
        sd.wait()  # Warten bis Aufnahme fertig
        audio = audio.flatten()
        
        # Klassifizieren
        klasse, konfidenz = klassifiziere(audio)
        print(f"   → Erkannt: {klasse} ({konfidenz:.1%} sicher)\n")

except KeyboardInterrupt:
    print("\nGestoppt.")


